In [6]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera.pandas as pa  # new import
from pandera import Column, DataFrameSchema, Check
from common.loaders import load_valid_country_ids


# ======================
# Funciones de validación
# ======================

def is_valid_url_or_null(series: pd.Series) -> pd.Series:
    """Valida que cada valor sea una URL válida o esté vacío/nulo."""
    return series.fillna("").apply(
        lambda x: (x == "") or str(x).startswith(("http://", "https://"))
    )


# ======================
# Definición de esquema
# ======================

images_schema = DataFrameSchema({
    "cca3": Column(
        str,
        [
            Check.str_length(3, 3),
            Check.isin(load_valid_country_ids())
        ],
        nullable=False
    ),
    "flag": Column(str, nullable=True),
    "flag_png": Column(str, [Check(is_valid_url_or_null)], nullable=True),
    "flag_svg": Column(str, [Check(is_valid_url_or_null)], nullable=True),
    "flag_alt": Column(str, nullable=True),
    "coa_png": Column(str, [Check(is_valid_url_or_null)], nullable=True),
    "coa_svg": Column(str, [Check(is_valid_url_or_null)], nullable=True),
})


# ======================
# Tareas
# ======================

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,coatOfArms,flag,flags"


@task
def extract_images():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()


@task
def transform_images(data):
    df = pd.DataFrame(data)

    # Extraer campos útiles de flags y coatOfArms
    df["flag_png"] = df["flags"].apply(lambda x: x.get("png") if x else None)
    df["flag_svg"] = df["flags"].apply(lambda x: x.get("svg") if x else None)
    df["flag_alt"] = df["flags"].apply(lambda x: x.get("alt") if x else None)

    df["coa_png"] = df["coatOfArms"].apply(lambda x: x.get("png") if x else None)
    df["coa_svg"] = df["coatOfArms"].apply(lambda x: x.get("svg") if x else None)

    df_clean = df[["cca3", "flag", "flag_png", "flag_svg", "flag_alt", "coa_png", "coa_svg"]]
    return df_clean


@task
def validate_images(df: pd.DataFrame) -> pd.DataFrame:
    return images_schema.validate(df)


@task
def load_images(df: pd.DataFrame):
    file_path = Path("../stage/country_images.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")


# ======================
# Flow principal
# ======================

@flow(name="etl_images_flow")
def etl_images():
    data = extract_images()
    df = transform_images(data)
    df = validate_images(df)
    load_images(df)


if __name__ == "__main__":
    etl_images()


04:16:20.194 | INFO    | Flow run 'attentive-mink' - Beginning flow run 'attentive-mink' for flow 'etl_images_flow'

04:16:21.602 | INFO    | Task run 'extract_images-507' - Finished in state Completed()

04:16:21.863 | INFO    | Task run 'transform_images-333' - Finished in state Completed()

04:16:22.098 | INFO    | Task run 'validate_images-007' - Finished in state Completed()

Saved 250 rows to ..\stage\country_images.csv


04:16:22.325 | INFO    | Task run 'load_images-31c' - Finished in state Completed()

04:16:22.344 | INFO    | Flow run 'attentive-mink' - Finished in state Completed()